# 01 — Setup S3 Data Lake for Yelp Sentiment MLOps

This notebook is the entry point for the AWS portion of the Yelp Review Sentiment MLOps project.

By the end of this notebook you will have:

1. Verified your SageMaker / AWS identity.
2. Created a dedicated S3 bucket for the Yelp sentiment project.
3. Uploaded the raw Yelp Open Dataset (`Yelp-JSON.zip`) into the bucket under `raw/`.
4. Streamed the review JSON out of the nested zip/tar archive and written a sample of raw reviews to CSV.
5. Uploaded the raw-review CSV to S3 under `raw/reviews/` so that Athena and the next notebooks can query it.

This mirrors the structure of the AAI-508 Heart Valve project's `01_setup_S3_bucket.ipynb`.

In [2]:
import sys
!{sys.executable} -m pip install "sagemaker<3.0.0" --upgrade

  Using cached sagemaker-2.257.3-py3-none-any.whl.metadata (17 kB)


  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached packaging-24.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached pathos-0.3.5-py3-none-any.whl.metadata (11 kB)


  Using cached sagemaker_core-1.0.78-py3-none-any.whl.metadata (4.9 kB)


  Using cached ppft-1.7.8-py3-none-any.whl.metadata (12 kB)
  Using cached pox-0.3.7-py3-none-any.whl.metadata (8.0 kB)


Using cached sagemaker-2.257.3-py3-none-any.whl (1.7 MB)
Using cached attrs-25.4.0-py3-none-any.whl (67 kB)
Using cached packaging-24.2-py3-none-any.whl (65 kB)
Using cached sagemaker_core-1.0.78-py3-none-any.whl (444 kB)
Using cached pathos-0.3.5-py3-none-any.whl (82 kB)
Using cached pox-0.3.7-py3-none-any.whl (29 kB)
Using cached ppft-1.7.8-py3-none-any.whl (56 kB)


...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━ 6/7 [sagemaker]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━ 6/7 [sagemaker]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━ 6/7 [sagemaker]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━ 6/7 [sagemaker]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [sagemaker]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
autogluon-multimodal 1.5.0 requires fsspec[http]<=2025.3, but you have fsspec 2026.3.0 which is incompatible.
sagemaker-studio-analytics-extension 0.3.0 requires sparkmagic==0.22.0, but you have sparkmagic 0.21.0 which is incompatible.
snowflake-connector-python 3.17.4 requires cffi<2.0.0,>=1.9, but you have cffi 2.0.0 which is incompatible.
sparkmagic 0.21.0 requires pandas<2.0.0,>=0.17.1, but you have pandas 2.3.3 which i

In [1]:
import sagemaker

print(sagemaker)
print(sagemaker.__file__)
print(getattr(sagemaker, "__version__", "No version"))


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


<module 'sagemaker' from '/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py'>
/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py
2.257.3


In [2]:
import boto3
import sagemaker

sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name
account_id = boto3.client("sts").get_caller_identity()["Account"]

print("Region:    ", region)
print("Account ID:", account_id)
print("Role:      ", role)

Region:     us-east-1
Account ID: 965705611982
Role:       arn:aws:iam::965705611982:role/LabRole


## Define the bucket and prefixes

We use a deterministic bucket name that includes your AWS account ID so the bucket is globally unique. The prefix layout below mirrors what you will see in the `docs/aws_mlops_plan.md` document and follows the same `raw/`, `processed/`, `features/`, `models/`, `reports/` convention used in the Heart Valve project.

In [3]:
bucket = f"yelp-sentiment-mlops-{account_id}"
raw_prefix = "raw"
raw_zip_key = f"{raw_prefix}/yelp-json.zip"
raw_reviews_prefix = f"{raw_prefix}/reviews"
athena_staging_prefix = "athena/staging"

print("Bucket:                 ", bucket)
print("Raw zip key:            ", raw_zip_key)
print("Raw reviews CSV prefix: ", raw_reviews_prefix)
print("Athena staging prefix:  ", athena_staging_prefix)

%store bucket
%store raw_prefix
%store raw_zip_key
%store raw_reviews_prefix
%store athena_staging_prefix
%store region
%store account_id

Bucket:                  yelp-sentiment-mlops-965705611982
Raw zip key:             raw/yelp-json.zip
Raw reviews CSV prefix:  raw/reviews
Athena staging prefix:   athena/staging
Stored 'bucket' (str)
Stored 'raw_prefix' (str)
Stored 'raw_zip_key' (str)
Stored 'raw_reviews_prefix' (str)
Stored 'athena_staging_prefix' (str)
Stored 'region' (str)
Stored 'account_id' (str)


## Create the S3 bucket if it does not exist

In [4]:
s3 = boto3.client("s3", region_name=region)

def ensure_bucket(bucket_name: str, region_name: str) -> None:
    existing = {b["Name"] for b in s3.list_buckets().get("Buckets", [])}
    if bucket_name in existing:
        print(f"Bucket already exists: {bucket_name}")
        return
    if region_name == "us-east-1":
        s3.create_bucket(Bucket=bucket_name)
    else:
        s3.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={"LocationConstraint": region_name},
        )
    print(f"Created bucket: {bucket_name}")

ensure_bucket(bucket, region)

Created bucket: yelp-sentiment-mlops-965705611982


## Upload `Yelp-JSON.zip` to S3 (one-time, from your local machine)

The full Yelp Open Dataset is too large to ship through GitHub (4 GB compressed). You only need to upload it **once** from your local machine using the AWS CLI:

```bash
# Run this on your local laptop, not inside SageMaker
aws s3 cp data/Yelp-JSON.zip s3://<bucket-name-printed-above>/raw/yelp-json.zip
```

The cell below verifies the upload completed before continuing. If it has not been uploaded yet, run the command on your laptop and re-run this cell.

In [6]:
from botocore.exceptions import ClientError

EXPECTED_MIN_BYTES = 3 * 1024 ** 3  # 3 GB safety floor; the real Yelp-JSON.zip is ~4 GB

try:
    head = s3.head_object(Bucket=bucket, Key=raw_zip_key)
    size_bytes = head["ContentLength"]
    size_gb = size_bytes / (1024 ** 3)
    print(f"Found s3://{bucket}/{raw_zip_key} ({size_gb:.2f} GB / {size_bytes:,} bytes)")
    if size_bytes < EXPECTED_MIN_BYTES:
        raise RuntimeError(
            f"S3 object is too small ({size_gb:.2f} GB). The Yelp-JSON.zip should be ~4 GB. "
            f"The upload from your laptop was truncated. Re-upload with:\n"
            f"  aws s3 cp data/Yelp-JSON.zip s3://{bucket}/{raw_zip_key} --expected-size 4345335132"
        )
except ClientError as exc:
    print("NOT FOUND yet. Upload from your laptop with:")
    print(f"  aws s3 cp data/Yelp-JSON.zip s3://{bucket}/{raw_zip_key}")
    raise

Found s3://yelp-sentiment-mlops-965705611982/raw/yelp-json.zip (4.05 GB / 4,345,335,132 bytes)


## Stream the review JSON Lines directly out of the S3 zip

The Yelp Open Dataset archive is structured as:

```
Yelp-JSON.zip
  └── Yelp JSON/yelp_dataset.tar
          └── yelp_academic_dataset_review.json   (JSON Lines)
```

We use `s3fs` to open the S3 zip as a seekable file-like object, then walk into the inner tar.gz and stream JSON Lines through `tarfile`. **Nothing is downloaded to local disk.** This keeps SageMaker Studio's home volume free of the 4 GB archive and is much faster because the bytes stay inside AWS.

`MAX_RAW_REVIEWS` controls how many raw reviews we pull; 300,000 comfortably exceeds the rubric requirement of 10,000 records per class once neutral 3-star reviews are filtered out and the classes are balanced in notebook 03.

In [7]:
!pip install --disable-pip-version-check --quiet s3fs

import csv
import json
import os
import tarfile
import zipfile
from pathlib import Path

import s3fs

MAX_RAW_REVIEWS = int(os.environ.get("YELP_MAX_REVIEWS", "300000"))
REVIEW_MEMBER_SUFFIX = "yelp_academic_dataset_review.json"

LOCAL_DATA_DIR = Path("/home/sagemaker-user/yelp-sentiment-mlops-pipeline/data")
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_CSV_PATH = LOCAL_DATA_DIR / "reviews_raw.csv"

print(f"Will extract up to {MAX_RAW_REVIEWS:,} raw reviews directly from S3 (no local zip download).")

Will extract up to 300,000 raw reviews directly from S3 (no local zip download).


In [8]:
CSV_COLUMNS = ["review_id", "business_id", "user_id", "stars", "review_text", "date"]

def normalize_review(review: dict) -> dict:
    return {
        "review_id": review.get("review_id", ""),
        "business_id": review.get("business_id", ""),
        "user_id": review.get("user_id", ""),
        "stars": review.get("stars", ""),
        "review_text": (review.get("text") or "").replace("\n", " ").replace("\r", " "),
        "date": review.get("date", ""),
    }

def stream_reviews_from_s3(s3_uri: str, csv_path: Path, max_reviews: int) -> int:
    fs = s3fs.S3FileSystem()
    written = 0
    with fs.open(s3_uri, mode="rb") as s3_file:
        with zipfile.ZipFile(s3_file) as zf:
            tar_name = next(
                name for name in zf.namelist()
                if name.endswith(".tar") and not name.startswith("__MACOSX/")
            )
            print(f"Streaming inner archive: {tar_name}")
            with zf.open(tar_name) as tar_stream:
                with tarfile.open(fileobj=tar_stream, mode="r|gz") as tf:
                    with csv_path.open("w", newline="", encoding="utf-8") as out_file:
                        writer = csv.DictWriter(out_file, fieldnames=CSV_COLUMNS)
                        writer.writeheader()
                        for member in tf:
                            if not member.name.endswith(REVIEW_MEMBER_SUFFIX):
                                continue
                            extracted = tf.extractfile(member)
                            if extracted is None:
                                break
                            for line in extracted:
                                review = json.loads(line.decode("utf-8"))
                                writer.writerow(normalize_review(review))
                                written += 1
                                if written % 25000 == 0:
                                    print(f"  wrote {written:,} reviews so far")
                                if written >= max_reviews:
                                    return written
                            break
    return written

s3_zip_uri = f"s3://{bucket}/{raw_zip_key}"
row_count = stream_reviews_from_s3(s3_zip_uri, LOCAL_CSV_PATH, MAX_RAW_REVIEWS)
print(f"Wrote {row_count:,} raw reviews to {LOCAL_CSV_PATH}")

Streaming inner archive: Yelp JSON/yelp_dataset.tar


  wrote 25,000 reviews so far


  wrote 50,000 reviews so far


  wrote 75,000 reviews so far


  wrote 100,000 reviews so far


...

  wrote 200,000 reviews so far


  wrote 225,000 reviews so far


  wrote 250,000 reviews so far


  wrote 275,000 reviews so far


  wrote 300,000 reviews so far
Wrote 300,000 raw reviews to /home/sagemaker-user/yelp-sentiment-mlops-pipeline/data/reviews_raw.csv


## Upload the raw-reviews CSV to S3

Athena will register an external table over this CSV in the next notebook.

In [9]:
raw_reviews_key = f"{raw_reviews_prefix}/reviews_raw.csv"
s3.upload_file(str(LOCAL_CSV_PATH), bucket, raw_reviews_key)
print(f"Uploaded to s3://{bucket}/{raw_reviews_key}")
%store raw_reviews_key

Uploaded to s3://yelp-sentiment-mlops-965705611982/raw/reviews/reviews_raw.csv
Stored 'raw_reviews_key' (str)


In [10]:
print(f"S3 contents under s3://{bucket}/:")
paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=bucket):
    for obj in page.get("Contents", []):
        size_mb = obj["Size"] / (1024 ** 2)
        print(f"  {obj['Key']:<60} {size_mb:>10.2f} MB")

S3 contents under s3://yelp-sentiment-mlops-965705611982/:
  raw/reviews/reviews_raw.csv                                      185.59 MB
  raw/yelp-json.zip                                               4144.03 MB


## Done

Continue to `athena_queries/01_Create_Athena_Database.ipynb`.